# Hampel filter: anomaly dates vs real-world events (concept test)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmmtscpr3/onyx-rhyolite/blob/main/EarlyWarning/notebooks/hampel_anomaly_colab.ipynb)

Phase 4 idea, tested before building it into `ews/`:

1. Run a **Hampel filter** over the 30 PIHPS weekly food prices and the 24 BI SPIP monthly series.
2. Turn the flags into **anomaly dates**: spikes (up) and drops (down).
3. Compare them with **known real-world events**: which events were caught and how early, and which flags have no explanation yet.

The data is `EarlyWarning/data/observations.csv`, the Phase 1 output (validated, log-transformed). Everything runs top to bottom with **Runtime → Run all**. The cells to edit are **2. Settings** and **5. Known events**.

In [ ]:
import math
import fnmatch
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from numpy.lib.stride_tricks import sliding_window_view
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

## 1. Load the data

Read straight from GitHub (the repository is public). To test with your own copy, upload the file to Colab and set `DATA_URL = "observations.csv"`.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmmtscpr3/onyx-rhyolite/main/EarlyWarning/data/observations.csv"

obs = pd.read_csv(DATA_URL, parse_dates=["ref_date"], low_memory=False)
obs["source"] = np.where(obs["series_id"].str.startswith("pihps."), "pihps", "bi")
# Flows (card, e-money, RTGS value and volume) are per day: x_cal = value / days in month.
obs["is_flow"] = ~np.isclose(obs["x_cal"], obs["value"], equal_nan=True)

print(f"{len(obs):,} rows, {obs['series_id'].nunique()} series, latest period {obs['ref_date'].max():%Y-%m-%d}")
display(obs.groupby(["source", "frequency"]).agg(series=("series_id", "nunique"),
                                                 first=("ref_date", "min"), last=("ref_date", "max")))

## 2. Settings

| Setting | Meaning |
|---|---|
| `window` | Periods in the baseline. Trailing: the `window` periods **before** the point. Centered: `window // 2` on each side plus the point. |
| `n_sigma` | Flag a point more than this many σ from the window's median. 3 is the textbook value. |
| `floor` | Minimum σ, in log units (0.01 ≈ 1%). Stops flat stretches (MAD = 0, common for rice, beef, sugar and cooking oil) from flagging every tiny move. |
| `centered` | `False`: only past data, as a live early-warning system would see it. `True`: the textbook filter (and the `hampel` package), which needs `window // 2` periods of future data. |
| `on` | `"level"`: far from the recent level. `"change"`: an unusual jump from the previous period. |

In [ ]:
SETTINGS = {
    "weekly":  dict(window=8,  n_sigma=3.5, floor=0.01, centered=False, on="level"),   # PIHPS
    "monthly": dict(window=12, n_sigma=3.5, floor=0.01, centered=False, on="level"),   # BI SPIP
}

## 3. The Hampel filter

For each period, take its window and compute

- **median** of the window,
- **MAD** = median of `|window − median|`,
- **σ = max(1.4826 × MAD, floor)**. 1.4826 = 1 / Φ⁻¹(0.75) makes the MAD read like a standard deviation for normal data,

then flag the period when `|x − median| > n_sigma × σ`. It runs on `x`, the log of each series, so a σ of 0.01 is about 1% in every series.

A window needs at least 75% of its periods present; missing periods are never flagged.

In [ ]:
MAD_TO_SIGMA = 1.4826
MIN_VALID_SHARE = 0.75


def hampel(x, window=8, n_sigma=3.5, floor=0.01, centered=False, on="level"):
    """Hampel filter over one regularly spaced series (NaN for missing periods).

    Returns a DataFrame with the same index: signal, median, mad, sigma, deviation, z, flag, direction.
    """
    x = pd.Series(x, dtype=float)
    signal = x.diff() if on == "change" else x
    values = signal.to_numpy(dtype=float)
    n = len(values)

    if centered:
        half = window // 2
        padded = np.concatenate([np.full(half, np.nan), values, np.full(half, np.nan)])
        windows = sliding_window_view(padded, 2 * half + 1)[:n]      # row i = values[i-half : i+half+1]
    else:
        padded = np.concatenate([np.full(window, np.nan), values])
        windows = sliding_window_view(padded, window)[:n]            # row i = values[i-window : i]

    enough = np.sum(~np.isnan(windows), axis=1) >= math.ceil(MIN_VALID_SHARE * windows.shape[1])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)              # all-NaN windows at the start
        median = np.nanmedian(windows, axis=1)
        mad = np.nanmedian(np.abs(windows - median[:, None]), axis=1)  # distances from THIS window's median
    median = np.where(enough, median, np.nan)
    mad = np.where(enough, mad, np.nan)

    sigma = np.where(np.isnan(mad), np.nan, np.maximum(MAD_TO_SIGMA * mad, floor))
    deviation = values - median
    with np.errstate(divide="ignore", invalid="ignore"):
        z = np.where(sigma > 0, deviation / sigma, np.where(deviation == 0, 0.0, np.sign(deviation) * np.inf))
    flag = np.abs(deviation) > n_sigma * sigma                       # False wherever anything is missing
    direction = np.where(flag, np.where(deviation > 0, "up", "down"), "")
    return pd.DataFrame({"signal": values, "median": median, "mad": mad, "sigma": sigma,
                         "deviation": deviation, "z": z, "flag": flag, "direction": direction}, index=x.index)

Quick check on the textbook example: a window of `[100, 101, 99, 130, 100, 102, 101]` around 130. The median is 101, MAD = 1, σ = 1.48, and |130 − 101| = 29 > 3 × 1.48, so 130 is flagged.

In [ ]:
demo = hampel(pd.Series([100, 101, 99, 130, 100, 102, 101.0]), window=6, n_sigma=3, floor=0, centered=True)
demo.round(2)

## 4. Run it on every series

In [ ]:
def run_all(obs, settings=SETTINGS):
    parts = []
    for series_id, g in obs.groupby("series_id", sort=True):
        g = g.sort_values("ref_date")
        result = hampel(g["x"].reset_index(drop=True), **settings[g["frequency"].iloc[0]])
        result.index = g.index
        parts.append(pd.concat([g, result], axis=1))
    return pd.concat(parts, ignore_index=True)


detail = run_all(obs)
flags = detail[detail["flag"]].reset_index(drop=True)

rate = detail.groupby("source").apply(lambda d: d["flag"].sum() / d["signal"].notna().sum(), include_groups=False)
print(f"{len(flags):,} flags in {flags['series_id'].nunique()} series")
print("Share of periods flagged:", rate.map("{:.1%}".format).to_dict())

## 5. Known events (edit me)

From `Scope/KNOWN_EVENTS.md`. All rows are still **candidates**: fix the dates, add events you know about, and re-run from here.

- `series`: glob patterns on `series_id`, e.g. `pihps.*.06_bawang_putih` = garlic in all three markets.
- `direction`: `up`, `down` or `any`. A one-off event only counts as **caught** by a flag in this direction.
- **Recurring** events (holidays, seasons) should *not* alarm on their own. They explain flags but are not in the hit/miss report.

In [ ]:
ONE_OFF = [
    # id,   name,                                              start,        end,          series patterns,                                                    direction
    ("E1",  "Toll roads go fully cashless",                    "2017-10-01", "2017-12-31", "bi_emoney.instruments*;bi_emoney.volume*;bi_emoney.value*;bi_payment_system.emoney_instruments_outstanding", "up"),
    ("E2",  "Garlic price spike (import delays)",              "2019-04-01", "2019-05-31", "pihps.*.06_bawang_putih", "up"),
    ("E3",  "COVID-19 and social restrictions (PSBB)",         "2020-03-01", "2020-06-30", "bi_card_transactions.*;bi_emoney.*;bi_payment_system.rtgs.*", "down"),
    ("E4",  "Bird's-eye chili price spike",                    "2021-01-01", "2021-03-31", "pihps.*.08_cabai_rawit", "up"),
    ("E5",  "Cooking oil crisis: price caps then export ban",  "2021-11-01", "2022-06-30", "pihps.*.09_minyak_goreng", "any"),
    ("E6",  "VAT rise from 10% to 11%",                        "2022-04-01", "2022-04-30", "pihps.*", "up"),
    ("E7",  "Egg price spike",                                 "2022-08-01", "2022-08-31", "pihps.*.04_telur_ayam", "up"),
    ("E8",  "Fuel (BBM) price rise",                           "2022-09-03", "2022-09-30", "pihps.*", "up"),
    ("E9",  "Rice price surge (El Nino, low harvest)",         "2023-08-01", "2024-03-31", "pihps.*.01_beras", "up"),
    ("E10", "General election",                                "2024-02-14", "2024-02-29", "bi_payment_system.currency_in_circulation", "any"),
]

# Holiday dates (Indonesia). Each holiday's window: (days before, days after).
HOLIDAYS = {
    "idul_fitri": ["2009-09-20", "2010-09-10", "2011-08-31", "2012-08-19", "2013-08-08", "2014-07-28",
                   "2015-07-17", "2016-07-06", "2017-06-25", "2018-06-15", "2019-06-05", "2020-05-24",
                   "2021-05-13", "2022-05-02", "2023-04-22", "2024-04-10", "2025-03-31", "2026-03-20"],
    "idul_adha":  ["2019-08-11", "2020-07-31", "2021-07-20", "2022-07-10", "2023-06-29", "2024-06-17",
                   "2025-06-06", "2026-05-27"],
}
HOLIDAY_WINDOWS = {"idul_fitri": (-28, 14), "idul_adha": (-21, 7)}

RECURRING = [
    # id,      name,                      when (holiday or month-day range), series patterns
    ("R-IF",   "Ramadan and Idul Fitri",  "idul_fitri",    "pihps.*.02_daging_ayam;pihps.*.03_daging_sapi;pihps.*.04_telur_ayam;pihps.*.05_bawang_merah;pihps.*.07_cabai_merah;pihps.*.08_cabai_rawit;bi_payment_system.currency_in_circulation;bi_card_transactions.*;bi_emoney.*"),
    ("R-IA",   "Idul Adha",               "idul_adha",     "pihps.*.03_daging_sapi"),
    ("R-YE",   "Christmas and New Year",  "12-01..01-15",  "pihps.*;bi_card_transactions.*;bi_emoney.*;bi_payment_system.currency_in_circulation"),
    ("R-RAIN", "Rainy season",            "11-01..03-31",  "pihps.*.05_bawang_merah;pihps.*.07_cabai_merah;pihps.*.08_cabai_rawit"),
    ("R-HARV", "Rice harvest season",     "03-01..05-31",  "pihps.*.01_beras"),
]

In [ ]:
TOLERANCE = {"weekly": pd.Timedelta(weeks=4), "monthly": pd.Timedelta(days=31)}  # around one-off events
EPISODE_GAP = {"weekly": 4, "monthly": 2}     # flags this many periods apart or closer = one episode
TARGET_LAG = 2                                # "caught early" = within 2 periods of the start
EXPLANATIONS = ["data_issue", "one_off", "recurring", "unexplained"]


def expand_events(years):
    """One row per occurrence of every event."""
    rows = []
    for eid, name, start, end, series, direction in ONE_OFF:
        rows.append(dict(event_id=eid, occurrence=eid, kind="one_off", name=name, start=pd.Timestamp(start),
                         end=pd.Timestamp(end), patterns=series.split(";"), direction=direction))
    for eid, name, when, series in RECURRING:
        base = dict(event_id=eid, kind="recurring", name=name, patterns=series.split(";"), direction="any")
        if when in HOLIDAYS:
            before, after = HOLIDAY_WINDOWS[when]
            for day in pd.to_datetime(HOLIDAYS[when]):
                rows.append(dict(base, occurrence=f"{eid}:{day.year}",
                                 start=day + pd.Timedelta(days=before), end=day + pd.Timedelta(days=after)))
        else:
            first, last = when.split("..")
            for year in years:
                s, e = pd.Timestamp(f"{year}-{first}"), pd.Timestamp(f"{year}-{last}")
                if e < s:
                    e = pd.Timestamp(f"{year + 1}-{last}")
                rows.append(dict(base, occurrence=f"{eid}:{year}", start=s, end=e))
    return pd.DataFrame(rows)


def matches(series_id, patterns):
    return any(fnmatch.fnmatchcase(series_id, p) for p in patterns)


def period_end(frame):
    """Last day each period covers, so a monthly flag counts for its whole month."""
    weekly = frame["ref_date"] + pd.Timedelta(days=6)
    monthly = frame["ref_date"] + pd.offsets.MonthEnd(0)
    return weekly.where(frame["frequency"] == "weekly", monthly)


def label_flags(flags, occurrences):
    """Explain each flag, in this order: data issue > one-off event > recurring > unexplained."""
    out = flags.copy()
    ends = period_end(out)
    tolerance = out["frequency"].map(TOLERANCE)
    ids, names, kinds = ([[] for _ in range(len(out))] for _ in range(3))
    all_series = out["series_id"].unique()
    for occ in occurrences.itertuples(index=False):
        hit = [s for s in all_series if matches(s, occ.patterns)]
        if not hit:
            continue
        same_series = out["series_id"].isin(hit)
        slack = tolerance if occ.kind == "one_off" else pd.Timedelta(0)
        overlap = (out["ref_date"] <= occ.end + slack) & (ends >= occ.start - slack)
        for i in np.flatnonzero(same_series & overlap):
            ids[i].append(occ.occurrence); names[i].append(occ.name); kinds[i].append(occ.kind)
    out["event_ids"] = [";".join(v) for v in ids]
    out["event_names"] = ["; ".join(dict.fromkeys(v)) for v in names]
    data_issue = out["quality_flag"].isin(["review", "masked_error"])
    out["explanation"] = [
        "data_issue" if d else "one_off" if "one_off" in k else "recurring" if k else "unexplained"
        for d, k in zip(data_issue, kinds)]
    return out


def periods_between(start, later, frequency):
    if frequency == "weekly":
        return int(round((later - start).days / 7))
    return (later.year - start.year) * 12 + (later.month - start.month)


def episodes(flags):
    """Runs of flags in one series, at most EPISODE_GAP periods apart."""
    rows = []
    for series_id, g in flags.sort_values("ref_date").groupby("series_id"):
        freq = g["frequency"].iloc[0]
        run = []
        for r in g.itertuples(index=False):
            if run and periods_between(run[-1].ref_date, r.ref_date, freq) > EPISODE_GAP[freq]:
                rows.append(_episode(series_id, freq, run)); run = []
            run.append(r)
        if run:
            rows.append(_episode(series_id, freq, run))
    return pd.DataFrame(rows)


def _episode(series_id, freq, run):
    dirs = {r.direction for r in run}
    expl = {r.explanation for r in run}
    return dict(series_id=series_id, frequency=freq, start=run[0].ref_date, end=run[-1].ref_date,
                n_flags=len(run), direction=dirs.pop() if len(dirs) == 1 else "mixed",
                peak_z=max((r.z for r in run), key=abs),
                explanation=next(e for e in EXPLANATIONS if e in expl),
                events="; ".join(dict.fromkeys(n for r in run for n in r.event_names.split("; ") if n)))


def event_report(flags, occurrences, detail):
    """Hit/miss per one-off event: caught when a matching series has a flag in the expected direction
    inside the event's dates (give or take TOLERANCE). lag = periods from the start to the first flag."""
    series = detail[["series_id", "frequency"]].drop_duplicates()
    rows = []
    for occ in occurrences[occurrences["kind"] == "one_off"].itertuples(index=False):
        expected = series[series["series_id"].map(lambda s: matches(s, occ.patterns))]
        if expected.empty:
            continue
        freq = expected["frequency"].mode().iloc[0]
        f = flags[flags["series_id"].isin(expected["series_id"])]
        f = f[(period_end(f) >= occ.start - TOLERANCE[freq]) & (f["ref_date"] <= occ.end + TOLERANCE[freq])]
        if occ.direction != "any":
            f = f[f["direction"] == occ.direction]
        first = f["ref_date"].min() if len(f) else pd.NaT
        lag = periods_between(occ.start, first, freq) if len(f) else None
        rows.append(dict(event=f"{occ.event_id} {occ.name}", start=occ.start.date(), end=occ.end.date(),
                         expected=occ.direction, result="caught" if len(f) else "MISSED",
                         series_caught=f"{f['series_id'].nunique()} / {len(expected)}",
                         first_flag=first.date() if len(f) else None,
                         lag=lag, unit="weeks" if freq == "weekly" else "months",
                         within_target=lag is not None and lag <= TARGET_LAG))
    return pd.DataFrame(rows)

### Hit/miss on the known events

`lag` counts periods from the event's listed start to the first flag; negative means before the start, which often means the start date needs checking.

In [ ]:
years = range(detail["ref_date"].dt.year.min() - 1, detail["ref_date"].dt.year.max() + 1)
occurrences = expand_events(years)
flags = label_flags(flags, occurrences)
eps = episodes(flags)
report = event_report(flags, occurrences, detail)

caught = report["result"].eq("caught")
print(f"Caught {caught.sum()} of {len(report)} one-off events; {report['within_target'].sum()} within {TARGET_LAG} periods of the start.")
print("Flags by explanation:", flags["explanation"].value_counts().reindex(EXPLANATIONS, fill_value=0).to_dict())
report

## 6. Look at one series

Grey band: inside it nothing is flagged (median ± n σ). ▲ above the band, ▼ below it, coloured by explanation. Orange areas: one-off events that should move this series.

Series IDs: `pihps.<traditional|modern|wholesale>.<01_beras … 10_gula_pasir>` and the BI ones such as `bi_emoney.volume` (`sorted(obs.series_id.unique())` lists them all).

In [ ]:
COLOURS = {"data_issue": "#8a8f98", "one_off": "#e8871e", "recurring": "#2a9d8f", "unexplained": "#d62839"}


def plot_series(series_id, settings=None, shade_recurring=False):
    g = obs[obs["series_id"] == series_id].sort_values("ref_date").reset_index(drop=True)
    freq = g["frequency"].iloc[0]
    s = {**SETTINGS[freq], **(settings or {})}
    r = pd.concat([g, hampel(g["x"], **s)], axis=1)
    explained = label_flags(r[r["flag"]].reset_index(drop=True), occurrences)[["ref_date", "explanation"]]
    r = r.merge(explained, on="ref_date", how="left")
    to_shown = np.exp if s["on"] == "level" else (lambda v: 100 * (np.exp(v) - 1))
    width = s["n_sigma"] * r["sigma"]
    shown, lo, hi, mid = (to_shown(v) for v in (r["signal"], r["median"] - width, r["median"] + width, r["median"]))

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=r["ref_date"], y=hi, line=dict(width=0), hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(x=r["ref_date"], y=lo, line=dict(width=0), fill="tonexty",
                             fillcolor="rgba(120,130,150,0.2)", name=f"normal band (±{s['n_sigma']:g} σ)", hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=r["ref_date"], y=mid, name="window median", line=dict(color="grey", width=1, dash="dot")))
    fig.add_trace(go.Scatter(x=r["ref_date"], y=shown, name=series_id, line=dict(color="#3a6ea5", width=1.6)))
    for expl, f in r[r["flag"]].groupby("explanation"):
        fig.add_trace(go.Scatter(
            x=f["ref_date"], y=shown[f.index], mode="markers", name=expl,
            marker=dict(color=COLOURS[expl], size=9, line=dict(width=0.5, color="white"),
                        symbol=["triangle-up" if d == "up" else "triangle-down" for d in f["direction"]]),
            customdata=f[["z"]], hovertemplate="%{x|%d %b %Y}: z = %{customdata[0]:.1f}<extra></extra>"))
    mine = occurrences[occurrences["patterns"].map(lambda p: matches(series_id, p))
                       & (occurrences["end"] >= r["ref_date"].min()) & (occurrences["start"] <= r["ref_date"].max())]
    for occ in mine.itertuples(index=False):
        if occ.kind == "one_off":
            fig.add_vrect(x0=occ.start, x1=occ.end, fillcolor="#e8871e", opacity=0.13, line_width=0,
                          annotation_text=occ.event_id, annotation_position="top left")
        elif shade_recurring:
            fig.add_vrect(x0=occ.start, x1=occ.end, fillcolor="#2a9d8f", opacity=0.07, line_width=0)
    unit = g["unit"].iloc[0] + (" per day" if g["is_flow"].iloc[0] else "")
    fig.update_layout(title=f"{series_id}: {int(r['flag'].sum())} flags  ({'centered' if s['centered'] else 'trailing'} "
                            f"{s['window']}, {s['n_sigma']:g} σ, floor {s['floor']:.1%}, on {s['on']})",
                      yaxis_title=unit if s["on"] == "level" else "% change", height=450,
                      hovermode="x unified", legend=dict(orientation="h", y=-0.15), margin=dict(t=50))
    fig.show()


plot_series("pihps.traditional.01_beras")

In [ ]:
plot_series("pihps.wholesale.04_telur_ayam")
plot_series("bi_card_transactions.credit.volume", shade_recurring=True)

### Try settings interactively

Pick a series and move the sliders; the chart re-runs the filter on that series only.

In [ ]:
import ipywidgets as w

w.interact(
    lambda series_id, window, n_sigma, floor_pct, centered, on: plot_series(
        series_id, dict(window=window, n_sigma=n_sigma, floor=floor_pct / 100, centered=centered, on=on)),
    series_id=w.Dropdown(options=sorted(obs["series_id"].unique()), value="pihps.traditional.08_cabai_rawit"),
    window=w.IntSlider(value=8, min=4, max=36, description="window"),
    n_sigma=w.FloatSlider(value=3.5, min=2, max=6, step=0.5, description="n σ"),
    floor_pct=w.FloatSlider(value=1.0, min=0, max=5, step=0.25, description="floor %"),
    centered=w.Checkbox(value=False, description="centered"),
    on=w.Dropdown(options=["level", "change"], value="level"),
);

## 7. Unexplained episodes

Runs of flags no known event, holiday or data issue explains. Each is either a **real event missing from the list** (add it in section 5) or a **false alarm** (the settings are too sensitive).

In [ ]:
unexplained = eps[eps["explanation"] == "unexplained"].sort_values(["n_flags", "start"], ascending=[False, False])
print(f"{len(unexplained)} unexplained episodes out of {len(eps)}")
display(unexplained.head(25).assign(peak_z=lambda d: d["peak_z"].round(1)))

Episodes on the same dates across several series or markets are more likely to be real. Grouped by commodity (PIHPS) or dataset (BI) and month:

In [ ]:
u = flags[flags["explanation"] == "unexplained"].copy()
u["group"] = np.where(u["source"] == "pihps", u["series_id"].str.split(".").str[2], u["series_id"].str.split(".").str[0])
u["month"] = u["ref_date"].dt.to_period("M")
together = (u.groupby(["group", "month", "direction"])
             .agg(series=("series_id", "nunique"), flags=("z", "size"), max_abs_z=("z", lambda z: z.abs().max()))
             .reset_index().query("series >= 2").sort_values(["series", "flags"], ascending=False))
together.head(25)

## 8. Overview: flags per series and year

In [ ]:
for source, f in flags.groupby("source"):
    grid = f.assign(year=f["ref_date"].dt.year).pivot_table(index="series_id", columns="year", values="z",
                                                             aggfunc="size", fill_value=0)
    fig = px.imshow(grid, color_continuous_scale="Reds", aspect="auto", text_auto=True,
                    title=f"{source}: flags per series and year", height=max(350, 22 * len(grid)))
    fig.update_xaxes(type="category")
    fig.show()

## 9. Compare settings

The concept question: which settings catch the known events early **without** too many unexplained flags? Each row re-runs everything with one combination, changing one thing at a time from the defaults in section 2.

- **Window:** 6 to 13 weeks all do about as well. Shorter windows lean on the floor; longer ones keep flagging after a price settles at a new level.
- **Centered** flags far less and catches few events: with future data in the window, the median moves with a sustained surge, so only isolated spikes stand out. The trailing window suits event detection.
- The threshold (`n_sigma`, `floor`) moves the flag rate more than the window does.

In [ ]:
def score(settings):
    d = run_all(obs, settings)
    f = label_flags(d[d["flag"]].reset_index(drop=True), occurrences)
    rep = event_report(f, occurrences, d)
    return dict(flag_rate=f"{d['flag'].sum() / d['signal'].notna().sum():.1%}",
                events_caught=f"{rep['result'].eq('caught').sum()} / {len(rep)}",
                within_target=int(rep["within_target"].sum()),
                unexplained_share=f"{f['explanation'].eq('unexplained').mean():.0%}")


def variant(weekly=None, monthly=None, **common):
    """SETTINGS with some values changed: weekly/monthly = window sizes, common = applies to both."""
    s = {freq: {**SETTINGS[freq], **common} for freq in SETTINGS}
    if weekly:
        s["weekly"]["window"] = weekly
    if monthly:
        s["monthly"]["window"] = monthly
    return s


CANDIDATES = {
    "defaults": variant(),
    "window 4 wk / 6 mo": variant(weekly=4, monthly=6),
    "window 13 wk / 24 mo": variant(weekly=13, monthly=24),
    "window 26 wk / 36 mo": variant(weekly=26, monthly=36),
    "n_sigma 3.0": variant(n_sigma=3.0),
    "n_sigma 4.0": variant(n_sigma=4.0),
    "floor 0 (none)": variant(floor=0.0),
    "floor 2%": variant(floor=0.02),
    "n_sigma 4.0, floor 2%": variant(n_sigma=4.0, floor=0.02),
    "centered (textbook)": variant(centered=True),
    "on change": variant(on="change"),
}
results = []
for name, s in CANDIDATES.items():
    results.append(dict(settings=name, **score(s)))
pd.DataFrame(results)

## 10. Compared with the `hampel` package (optional)

`hampel(data, window_size=12, n_sigma=3.0)` from PyPI is the **centered** filter with **no floor**:

- `window_size` is the *total* window: it uses `window_size // 2` points on each side, so 12 gives 13 points (6 before, the point, 6 after) and needs 6 periods of **future** data.
- σ = 1.4826 × MAD with no minimum, so on a flat stretch (MAD = 0) any change at all is flagged.
- The first and last `window_size // 2` points are never tested, and its `medians` / `thresholds` arrays hold leftover values there, not NaN.
- It can't take NaN (gaps), and `outlier_indices` are positions, not dates.

With the same settings (`centered=True, floor=0`), the function above gives the same flags away from the ends:

In [ ]:
!pip -q install hampel
from hampel import hampel as hampel_pkg

g = obs[obs["series_id"] == "pihps.traditional.09_minyak_goreng"].sort_values("ref_date").reset_index(drop=True)
values = g["value"].interpolate().bfill()              # the package can't take gaps
theirs = set(hampel_pkg(values, window_size=12, n_sigma=3.0).outlier_indices.tolist())
ours = set(np.flatnonzero(hampel(values, window=12, n_sigma=3.0, floor=0, centered=True)["flag"]))
inside = set(range(6, len(values) - 6))
print("package:", len(theirs), "flags   ours:", len(ours & inside), "away from the ends   identical:", theirs & inside == ours & inside)

## 11. Save the results

In [ ]:
cols = ["series_id", "source", "frequency", "ref_date", "value", "unit", "direction", "z", "quality_flag",
        "explanation", "event_ids", "event_names"]
flags[cols].to_csv("hampel_flags.csv", index=False)
eps.to_csv("hampel_episodes.csv", index=False)
report.to_csv("hampel_event_report.csv", index=False)
print("Saved hampel_flags.csv, hampel_episodes.csv, hampel_event_report.csv")
try:
    from google.colab import files
    for name in ("hampel_flags.csv", "hampel_episodes.csv", "hampel_event_report.csv"):
        files.download(name)
except ImportError:
    pass